By default this notebook plots the paper's summary in `analysis/paper_data/reacher_extrapolation_bin_summary.json`. To plot your own evaluation, run from the repository root:

```bash
pixi run python -m analysis.reacher_extrapolation --config conf/analysis/reacher_extrapolation.json --output logs/extrapolation
pixi run python -m analysis.reacher_extrapolation --output logs/extrapolation_summary --summarize --results logs/extrapolation
```

Then set `summary_file` below to the `bin_summary.json` in your summary directory. The paper layout requires all three methods across six bins. Run plotting on CPU from the repository root or `analysis/`. Nimbus Roman is used when installed; otherwise the default serif font is used. Exports use a fresh `logs/paper_figures/` directory.


In [ ]:
%matplotlib inline
import os
import sys
import json
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.transforms import offset_copy
from matplotlib.patches import Rectangle

root = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
os.chdir(root)
sys.path.insert(0, str(root))
from analysis.artifacts import create_output

for font in Path('/usr/share/fonts/urw-base35').glob('NimbusRoman-*.otf'):
    font_manager.fontManager.addfont(str(font))
try:
    font_manager.findfont('Nimbus Roman', fallback_to_default=False)
    font_family = 'Nimbus Roman'
except ValueError:
    font_family = 'serif'
    print('Nimbus Roman not found; using the default serif font, so text layout may differ from the paper.')
plt.rcdefaults()
plt.rcParams.update({'font.family': font_family, 'mathtext.fontset': 'stix',
                     'pdf.fonttype': 42, 'ps.fonttype': 42})

summary_file = Path('analysis/paper_data/reacher_extrapolation_bin_summary.json')


## Read saved results

Read only the files selected above; no model inference or simulation.


In [ ]:
rows = json.loads(summary_file.read_text())
methods = ['vanilla', 'privileged_physics', 'context']
if len(rows) != 18 or {(row['method'], row['bin']) for row in rows} != {(method, bin_id) for method in methods for bin_id in range(6)}:
    raise ValueError(f'{summary_file}: expected three methods across six bins')
rows = sorted(rows, key=lambda row: (methods.index(row['method']), row['bin']))
for row in rows:
    values = [row['lower'], row['upper'], row['center'], row['psnr'], row['iou_percent']]
    values += [row[key] for key in ['psnr_sem', 'iou_percent_sem'] if row[key] is not None]
    if not np.isfinite(values).all():
        raise ValueError(f'{summary_file}: non-finite bin statistics')
inputs = {'summary': str(summary_file.resolve())}
print('Methods:', methods, '| Bins:', len({row['bin'] for row in rows}))
print('Targets per bin:', sorted({row['count'] for row in rows}))

run_dir = create_output(root / 'logs/paper_figures' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'), {'inputs': inputs})


## Plot and export

Keep the paper layout and export PDF/PNG and plotted values into a fresh output directory.


In [ ]:
page_width, page_height = 600.654, 276.408
fig = plt.figure(figsize=(page_width / 72, page_height / 72))
styles = [('vanilla', 'DINO-WM', '#aad2c8'),
          ('privileged_physics', r'+ Privileged Physics$^{\ddag}$', '#42a5f5'),
          ('context', '+ PhysiCal (Ours)', '#ff647b')]
for left, metric, title, ticks in [(37.47, 'psnr', 'PSNR ↑', [15, 17.5, 20, 22.5, 25]),
                                   (338.3228372093, 'iou_percent', 'IoU (%) ↑', [15, 30, 45, 60])]:
    ax = fig.add_axes([left / page_width, 48.6 / page_height,
                       261.6111627907 / page_width, 156.396 / page_height])
    for row in rows:
        if row['method'] == 'vanilla' and row['in_domain']:
            ax.axvspan(row['lower'], row['upper'], color='gray', alpha=.12, linewidth=0, zorder=0)
    for method, label, color in styles:
        selected = [row for row in rows if row['method'] == method]
        x = np.asarray([row['center'] for row in selected])
        values = np.asarray([row[metric] for row in selected])
        sem = np.asarray([row[metric + '_sem'] if row[metric + '_sem'] is not None else 0 for row in selected])
        ax.fill_between(x, values - sem, values + sem, color=color, alpha=.25, linewidth=0, zorder=2)
        ax.plot(x, values, color=color, marker='o', markersize=6.5, linewidth=2.5, zorder=3)
    ax.margins(y=.15)
    ax.set_xticks([.008, .016, .024])
    ax.set_yticks(ticks)
    ax.tick_params(labelsize=17, length=0, pad=7)
    ax.tick_params(axis='x', pad=7.28125)
    for tick in ax.get_yticklabels():
        tick.set_transform(offset_copy(tick.get_transform(), fig=fig, y=-.140625, units='points'))
    ax.grid(linestyle='--', alpha=.7, linewidth=1)
    ax.set_axisbelow('line')
    ax.set_title(title, fontsize=20, pad=6)
    for spine in ax.spines.values():
        spine.set_color('#b0b0b0')
        spine.set_alpha(.8)
        spine.set_linewidth(1.5)
fig.text(214.842625 / page_width, 5.516875 / page_height, 'Reacher Joint Damping', fontsize=22)
for x, (_, label, color) in zip([36.9960625, 183.9960625, 402.9960625], styles):
    fig.add_artist(Line2D([x / page_width, (x + 28.8) / page_width],
                          [266.988 / page_height] * 2, color=color, linewidth=2.5))
    fig.add_artist(Line2D([(x + 14.4) / page_width], [266.988 / page_height],
                          color=color, marker='o', markersize=6.5))
    fig.add_artist(Rectangle((x / page_width, 260.688 / page_height),
                             28.8 / page_width, 12.6 / page_height,
                             facecolor=color, alpha=.25, linewidth=0, zorder=2))
    fig.text((x + 36) / page_width, 260.688 / page_height, label, fontsize=18)
for x, label, color, alpha in [(150.082, 'train-range', 'gray', .12),
                               (292.082, 'out-of-train-range', 'white', 1)]:
    fig.add_artist(Rectangle((x / page_width, 235.44 / page_height),
                             28.8 / page_width, 12.6 / page_height,
                             facecolor=(*plt.matplotlib.colors.to_rgb(color), alpha), edgecolor='0.6'))
    fig.text((x + 36) / page_width, 235.44 / page_height, label, fontsize=18)
for extension in ('pdf', 'png'):
    fig.savefig(run_dir / f'concat-reacher-psnr_iou-v2.{extension}', dpi=200)
(run_dir / 'bin_summary.json').write_text(json.dumps(rows, indent=2) + '\n')
plt.show()
print(run_dir)


Paper styling is retained; plotted values come from the selected inputs. PDF file bytes may differ because of metadata.
